# ch06 练习 —— 类型转换与字符串处理

数据：`data/device_defects.csv`（270 行 x 13 列）与 `data/device_defects_raw.csv`（脏格式版）

**每题都有 `assert` 验收块**。跑通即正确，跑不通的报错信息本身就说清了哪里错。
**不要翻答案版。**

| 题号 | 考什么 |
|---|---|
| TODO(1) | 用 `exclude="number"` 挑出字符串列（并验证 `include="object"` 已废弃） |
| TODO(2) | 类别去噪（空白问题） |
| TODO(3) | 类别语义合并（8 类 → 3 类） |
| TODO(4) | 缺陷类型两步标准化（10 类 → 6 类） |
| TODO(5) | 设备编号拆前缀 |
| TODO(6) | 前缀反推设备类型 |
| TODO(7) | 日期规整 + `format="mixed"` 解析 |
| TODO(8) | 正确转可空整型 |
| TODO(9) | `category` 内存压缩 |
| 综合 ① | 完整类型清洗链（类型锁定 → 类别标准化 → 日期解析） |
| 综合 ② | 数据质量报告 |

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

DATA = Path("data")
dev = pd.read_csv(DATA / "device_defects.csv")
raw = pd.read_csv(DATA / "device_defects_raw.csv", dtype=str)

# 同义映射表（课里会逐条验证它们为什么是必需的）
STATUS_MAP = {"未处理": "待处理", "处理完成": "已处理", "done": "已处理"}
DEFECT_MAP = {"渗油": "渗漏油", "漏油": "渗漏油", "异物搭挂": "异物"}
PREFIX2TYPE = {"CB": "断路器", "CT": "互感器", "IN": "绝缘子", "LA": "避雷器", "TR": "变压器"}
CAT_COLS = ["台区编号", "线路名称", "设备类型", "缺陷类型", "缺陷等级", "处理状态"]

print("pandas", pd.__version__)
print("dev", dev.shape, "| raw", raw.shape)

---

## 基础题

In [ ]:
# TODO(1)：挑出全部字符串列（用 exclude="number"），存入 str_cols
# 提示：DataFrame.select_dtypes(exclude="number").columns.tolist()
str_cols = ____
# ---- 验收 ----
assert isinstance(str_cols, list), "str_cols 应该是 list"
assert len(str_cols) == 9, f"应有 9 个字符串列，实际 {len(str_cols)}"
assert "发现日期" in str_cols and "处理状态" in str_cols

# 顺带验证：include="object" 在 pandas 3.0 仍能选到，但会报警告（已废弃）
import warnings as _w

with _w.catch_warnings(record=True) as caught:
    _w.simplefilter("always")
    legacy_cols = list(dev.select_dtypes("object").columns)
assert legacy_cols == str_cols, "兼容行为下两者应一致"
assert any("Pandas4Warning" in type(x.message).__name__ for x in caught), "应抛出废弃警告"
print("TODO(1) 通过 -> 字符串列", len(str_cols), "个；include='object' 已废弃但仍可用")

In [ ]:
# TODO(2)：给「处理状态」去噪：去首尾空白 + 去内部空白，结果存入 status_step1
# 提示：Series.str.strip() ；Series.str.replace(r"\s+", "", regex=True)
status_step1 = ____
# ---- 验收 ----
assert status_step1.nunique() == 6, f"去噪后应为 6 类，实际 {status_step1.nunique()}"
counts_step1 = status_step1.value_counts().to_dict()
assert counts_step1["已处理"] == 144, f"已处理应为 144（128+16），实际 {counts_step1.get('已处理')}"
assert counts_step1["处理中"] == 54, f"处理中应为 54（37+17），实际 {counts_step1.get('处理中')}"
assert "处理中 " not in counts_step1 and "已 处理" not in counts_step1, "空白还没去干净"
print("TODO(2) 通过 -> 8 类降到", status_step1.nunique(), "类：", counts_step1)

In [ ]:
# TODO(3)：把去噪结果按 STATUS_MAP 做语义合并，存入 status_final，应只剩 3 类
# 提示：Series.replace(字典)
status_final = ____
# ---- 验收 ----
assert status_final.nunique() == 3, f"合并后应为 3 类，实际 {status_final.nunique()}"
counts_final = status_final.value_counts().to_dict()
assert counts_final == {"已处理": 171, "处理中": 54, "待处理": 45}, f"分布不对：{counts_final}"
assert len(status_final) == len(dev), "合并不能改变行数"
assert status_final.isna().sum() == 0, "replace 不应该产生缺失（变出 NaN 说明用错成 map 了）"
print("TODO(3) 通过 -> 最终分布:", counts_final)

In [ ]:
# TODO(4)：缺陷类型两步标准化：先 strip 再去噪，然后按 DEFECT_MAP 合并，存入 defect_final
# 提示：一步写完：dev["缺陷类型"].str.strip().replace(DEFECT_MAP)
defect_final = ____
# ---- 验收 ----
assert dev["缺陷类型"].nunique() == 10, "原始应有 10 种写法"
assert defect_final.nunique() == 6, f"合并后应为 6 类，实际 {defect_final.nunique()}"
counts_defect = defect_final.value_counts().to_dict()
assert counts_defect["渗漏油"] == 61, f"渗漏油应为 61（43+11+7），实际 {counts_defect.get('渗漏油')}"
assert counts_defect["异物"] == 40, f"异物应为 40（29+11），实际 {counts_defect.get('异物')}"
assert counts_defect["锈蚀"] == 36, f"锈蚀应为 36（29+7），实际 {counts_defect.get('锈蚀')}"
assert defect_final.isna().sum() == 0
print("TODO(4) 通过 ->", counts_defect)

In [ ]:
# TODO(5)：从「设备编号」拆出前缀，存入 sn_prefix（形如 'TR' 'CB'）
# 提示：Series.str.split("_").str[0]
sn_prefix = ____
# ---- 验收 ----
assert set(sn_prefix.unique()) == {"CB", "CT", "IN", "LA", "TR"}, f"前缀集合不对：{set(sn_prefix.unique())}"
counts_prefix = sn_prefix.value_counts().to_dict()
assert counts_prefix["TR"] == 55, f"TR 应为 55，实际 {counts_prefix.get('TR')}"
assert counts_prefix["LA"] == 59, f"LA 应为 59，实际 {counts_prefix.get('LA')}"
print("TODO(5) 通过 -> 前缀分布:", counts_prefix)

---

## 进阶题

In [ ]:
# TODO(6)：用 PREFIX2TYPE 把前缀反推回设备类型，存入 recovered_type
# 提示：Series.map(字典)
recovered_type = ____
# ---- 验收 ----
assert recovered_type.isna().sum() == 0, "不应有未命中的前缀"
assert recovered_type.nunique() == 5
assert bool((recovered_type == dev["设备类型"]).all()), "反推结果与原始设备类型不一致"
print("TODO(6) 通过 -> 前缀映射与设备类型 270 行全一致")

In [ ]:
# TODO(7)：解析 raw 的「发现日期」：先把中文年月日换成 '-'，再用 format="mixed" 解析，存入 found_date
# 提示：链式四次 str.replace 把 年/月 换成 '-'、把 日 去掉；再 pd.to_datetime(..., format="mixed")
date_norm = ____
found_date = ____
# ---- 验收 ----
assert found_date.isna().sum() == 0, f"不应有解析失败，实际失败 {int(found_date.isna().sum())} 行"
assert str(found_date.dtype).startswith("datetime64"), f"dtype 不对：{found_date.dtype}"
assert found_date.min().year == 2026 and found_date.max().month == 6

# 对照：不做规整、直接 mixed 会失败 12 行（中文日期）
naive = pd.to_datetime(raw["发现日期"], format="mixed", errors="coerce")
assert int(naive.isna().sum()) == 12, f"直接 mixed 应失败 12 行，实际 {int(naive.isna().sum())}"
print("TODO(7) 通过 -> 解析 0 失败；直接 mixed 则失败", int(naive.isna().sum()), "行")

In [ ]:
# TODO(8)：把「负荷值」转成可空整型，存入 load_int（先四舍五入）
# 提示：Series.round().astype("Int64")   —— 大写 I 的可空整型
load_int = ____
# ---- 验收 ----
assert str(load_int.dtype) == "Int64", f"dtype 应为 Int64，实际 {load_int.dtype}"
assert int(load_int.isna().sum()) == 21, f"缺失应保留 21 个，实际 {int(load_int.isna().sum())}"
assert int(load_int.notna().sum()) == 249
first_valid = load_int.dropna().iloc[0]
assert isinstance(first_valid, (int, np.integer)) or float(first_valid).is_integer(),     f"取到的值 {first_valid!r} 不是整数语义"

# 直接 astype(int) 必须失败——证明 round 这一步不可省
try:
    dev["负荷值"].astype(int)
    raise AssertionError("astype(int) 不该成功（列里有缺失 + 小数）")
except AssertionError:
    raise
except Exception:
    pass
print("TODO(8) 通过 -> Int64 保留", int(load_int.isna().sum()), "个 <NA>，前 3 值:", load_int.head(3).tolist())

In [ ]:
# TODO(9)：把 CAT_COLS 里的列一次性转成 category，存入 dev_cat
# 提示：DataFrame.astype({列: "category" for 列 in CAT_COLS})
dev_cat = ____
# ---- 验收 ----
assert all(str(dev_cat[c].dtype) == "category" for c in CAT_COLS), "有列没转成 category"
before = dev.memory_usage(deep=True).sum()
after = dev_cat.memory_usage(deep=True).sum()
saved = 1 - after / before
assert saved > 0.6, f"内存应省 60% 以上，实际 {saved:.1%}"
assert abs(saved - 0.670) < 0.01, f"实测应约 67.0%，实际 {saved:.1%}"
assert int(dev_cat["台区编号"].cat.codes.max()) == 17, "台区编号应有 18 个类别（codes 0..17）"

# 陷阱验证：category 列插入新值会报错
probe = pd.Series(["变压器", "绝缘子"]).astype("category")
try:
    probe.loc[0] = "开关柜"
    raise AssertionError("category 列插入新类别本该报 TypeError")
except AssertionError:
    raise
except TypeError:
    pass
print(f"TODO(9) 通过 -> 内存 {before} -> {after} 字节，省 {saved:.1%}")

---

## 综合题

### 综合 ① 完整类型清洗链

把本章所有动作串成一条流水线，产出一张**可直接进建模**的表 `clean`：

1. 类别列标准化：`处理状态`（8→3 类）、`缺陷类型`（10→6 类）
2. 日期解析：把 `发现日期` 真正转成 `datetime`
3. 数值类型锁定：`负荷值` 保持 `float64`（要参与算术），`记录ID` 转 `Int64`
4. 显式派生：从 `设备编号` 拆出 `设备前缀`
5. 内存优化：字符串列转 `category`（放在最后一步）

要求 `clean` 相对原始表**只增不减行**（270 行）、**不产生新的缺失**。

In [ ]:
# TODO(10)：综合①：按上面 5 步产出 clean（只挖标准化、类型锁定、派生、内存优化四段）
# 提示：先 copy；类别列用 .str.strip().replace(...)；日期用 pd.to_datetime(format="mixed")；最后 astype({...:"category"})
clean = ____
clean["处理状态"] = ____
clean["缺陷类型"] = ____
clean["发现日期"] = ____
clean["记录ID"] = ____
clean["设备前缀"] = ____
clean = ____
# ---- 验收 ----
assert len(clean) == 270, f"行数不能变，实际 {len(clean)}"
assert str(clean["发现日期"].dtype).startswith("datetime64"), "发现日期没转成日期"
assert str(clean["处理状态"].dtype) == "category", "处理状态没转成 category"
assert clean["处理状态"].nunique() == 3
assert clean["缺陷类型"].nunique() == 6
assert "设备前缀" in clean.columns and clean["设备前缀"].nunique() == 5

# 清洗过程不能把非缺失变成缺失：只允许原有缺失继续存在
for col in ["处理状态", "缺陷类型"]:
    assert clean[col].isna().sum() == 0, f"{col} 出现了新的缺失"
assert int(clean["负荷值"].isna().sum()) == 21, "负荷值缺失数不能被改动"

print("综合① 通过 -> clean", clean.shape)
print("   处理状态:", clean["处理状态"].value_counts().to_dict())
print("   缺陷类型:", {k: int(v) for k, v in clean["缺陷类型"].value_counts().items()})
print("   内存:", clean.memory_usage(deep=True).sum(), "字节（原", dev.memory_usage(deep=True).sum(), "）")

### 综合 ② 数据质量报告

写一个函数，输入原始 `dev`，输出 5 个关键质量指标。这张报告就是竞赛里
"数据探索"那几分的交出物，也是给评委看的第一张表。

In [ ]:
# TODO(11)：综合②：产出 5 个质量指标存入 quality 字典
# 提示：用 dev 现算：字符串列数 / 类别最大脏度 / 日期是否已解析 / 缺失单元格 / 可压缩比例
str_col_count = ____
max_dirty = ____
date_parsed = ____
missing_cells = ____
compress_ratio = ____
quality = ____
# ---- 验收 ----
expect = {"字符串列数", "最大类别脏度", "日期已解析", "缺失单元格", "可压缩比例"}
assert set(quality) == expect, f"键不对。多 {set(quality) - expect}，缺 {expect - set(quality)}"
assert quality["字符串列数"] == 9
assert quality["最大类别脏度"] == 10, "缺陷类型原始 10 种写法，是脏度最高的列"
assert quality["日期已解析"] is False, "原始表里发现日期还是字符串"
assert quality["缺失单元格"] == 240, f"全表缺失单元格应为 240，实际 {quality['缺失单元格']}"
assert abs(quality["可压缩比例"] - 0.6698) < 0.001,     f"可压缩比例应约 0.6698，实际 {quality['可压缩比例']}"
print("综合② 通过 -> 数据质量报告：")
for key, value in quality.items():
    print(f"   {key:12s} = {value!r}")

---

### 复盘提问

**复盘问自己**（五题都能答上来才算过）：

1. `pd.to_numeric(s, errors="ignore")` 错在哪？为什么它比 `errors="raise"` 更危险？
2. `format="mixed"` 为什么解不了 `2026年5月3日`？正确姿势是什么？
3. `map` 和 `replace` 都能做映射，同义合并该用哪个？用错会看到什么现象？
4. `Int64` 比 `float64` 更省内存吗？实测差多少、为什么？
5. 为什么 `astype("category")` 必须放在清洗的**最后一步**？

答不上来的，回 `_notes/错题本.md` 记一笔。